# MRP / 05 — Conditional reversal: confirmation 2023-07-01 – 2025-12-31 (one look)

Protocol: [`MRP 003`](../../discussions/MNQ_Return_Prediction/003_claude_mrp02_conditional_reversal_preregistration.md) §5. Runs only with `04/summary.json`, an unchanged `04/candidates.csv` (SHA-256) and at least one candidate. Set your Lucid commission first: it sets the market cost used in discovery, so it **must equal the value 04 was run with** (default $0.50/side → 3 ticks).

Gates per frozen candidate: **C1** n ≥ 75 · **C2** net > 0 at market cost and day-block bootstrap 95% lower bound of total net > 0 · **C3** net > 0 in ≥ 2 of 3 periods (2023H2, 2024, 2025).

In [3]:
import json, os, sys, tempfile
import datetime as dt
from dataclasses import replace
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

SYNTHETIC = os.environ.get("MRP_SYNTHETIC") == "1"
if SYNTHETIC:   # separate folder: the reversal smoke data has a planted reversal and no FX effect
    os.environ["EQUITY_TRADING_DIR"] = os.environ.get("MRP_REV_SYNTH_DIR") or tempfile.mkdtemp(prefix="mrp_rev_")
    os.environ["MRP_REV_SYNTH_DIR"] = os.environ["EQUITY_TRADING_DIR"]
    os.environ.pop("DUKASCOPY_DIR", None)

REPO = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(REPO / "src"))
from equity_trading import mnq_predict as P
from equity_trading import reversal as V

CFG = P.SYNTH_CFG if SYNTHETIC else P.MRPConfig()
D = dt.date
OUT = Path("05"); OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40); pd.set_option("display.max_rows", 80)
print("engines", P.MRP_VERSION, V.REV_VERSION, "| synthetic" if SYNTHETIC else "| real data")
if SYNTHETIC and not (Path(os.environ["EQUITY_TRADING_DIR"]) / "Databento").exists():
    P.write_synthetic(os.environ["EQUITY_TRADING_DIR"], CFG.warmup_start, CFG.test[1], CFG, beta_jpy=0.0,
                      reversal_points=6.0, tails_df=4)


engines 0.1.1 0.1.0 | real data


In [4]:
COMMISSION_PER_SIDE = 0.50          # USD per contract per side -- must match 04
COMMISSION_CONFIRMED = True
RC = V.RevConfig(commission_per_side=COMMISSION_PER_SIDE)
S = json.loads(Path("04/summary.json").read_text())
if not COMMISSION_CONFIRMED:
    raise RuntimeError("Confirm the Lucid commission and set COMMISSION_CONFIRMED = True.")
if abs(S["market_cost_ticks"] - RC.market_cost_ticks) > 1e-9:
    raise RuntimeError(f"market cost {RC.market_cost_ticks} != discovery {S['market_cost_ticks']}: rerun 04 with the confirmed commission first.")
if not V.frozen_ok(Path("04/candidates.csv"), S["candidates_sha256"]):
    raise RuntimeError("04/candidates.csv changed after it was frozen.")
cands = pd.read_csv("04/candidates.csv")
if cands.empty:
    raise RuntimeError("No candidate passed discovery: the test window is not opened.")
display(cands)

RuntimeError: No candidate passed discovery: the test window is not opened.

In [ ]:
if SYNTHETIC:
    PERIODS = {"2021Q1": (D(2021, 1, 1), D(2021, 3, 31)), "2021Q2": (D(2021, 4, 1), D(2021, 6, 30))}
else:
    PERIODS = {"2023H2": (D(2023, 7, 1), D(2023, 12, 31)), "2024": (D(2024, 1, 1), D(2024, 12, 31)),
               "2025": (D(2025, 1, 1), D(2025, 12, 31))}
grid = P.load_grid(CFG.warmup_start, CFG.test[1], CFG)
sig2 = V.event_sigma(grid, None, CFG.refits_test, CFG.test[1], CFG)
bt = V.bar_table(grid, sig2, CFG)
bt = bt[(bt["trade_date"] >= CFG.test[0]) & (bt["trade_date"] <= CFG.test[1])]
rows, store = [], {}
for c in cands.itertuples():
    tr = V.trades(bt, c.k, int(c.d), int(c.h), c.session)
    store[c.config] = tr
    s = V.summarize(tr, RC.market_cost_ticks, PERIODS)
    bs = V.day_block_lower(tr, RC.market_cost_ticks)
    pos = sum(s[f"net_{p}"] > 0 for p in PERIODS)
    rows.append({"config": c.config, **s, "boot_lower_total": bs["lower_95"],
                 "net_realized": V.summarize(tr, RC.realized_cost_ticks)["net"], "periods_positive": pos,
                 "C1": s["n"] >= RC.min_trades_confirm, "C2": (s["net"] > 0) and (bs["lower_95"] > 0),
                 "C3": pos >= min(2, len(PERIODS))})
res = pd.DataFrame(rows); res["confirmed"] = res["C1"] & res["C2"] & res["C3"]
display(res.round(3))

In [ ]:
fig, ax = plt.subplots(figsize=(11, 3.6))
for c, tr in store.items():
    ax.plot(pd.to_datetime(tr["trade_date"]), (tr["gross_ticks"] - RC.market_cost_ticks).cumsum(), label=c)
ax.axhline(0, color="k", lw=.6); ax.legend(fontsize=7); ax.set_title("cumulative net ticks per contract, test window")
plt.tight_layout(); plt.savefig(OUT / "confirmation.png", dpi=110); plt.show()
summary = {"engines": [P.MRP_VERSION, V.REV_VERSION], "synthetic": SYNTHETIC, "commission_per_side": COMMISSION_PER_SIDE,
           "results": res.to_dict("records"), "confirmed": res.loc[res["confirmed"], "config"].tolist()}
(OUT / "summary.json").write_text(json.dumps(summary, indent=1, default=lambda o: bool(o) if isinstance(o, np.bool_) else float(o)))
print("confirmed:", summary["confirmed"])